# HDM Error Correction Thematic Notebook

Notebook-libreria del pipeline strict pure-exogenous, separato in celle per area tematica.

## Mappa Del Notebook

Le sezioni sono organizzate in questo ordine: import e costanti, dataclasses, helper interni, DTU10, data preparation, model fitting, eventi e metriche, orchestrazione, plotting.

## Imports And Constants

Import, costanti globali, stazioni e mapping di base.

In [1]:
from dataclasses import dataclass
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.graphics.gofplots import qqplot
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sktime.split import ExpandingWindowSplitter


STATIONS = {
    "Sonderborg": 26473,
    "Assens": 28366,
    "Bagenkop": 28548,
    "Kobenhavn": 30336,
    "Koge": 30478,
    "Gedser": 31616,
}

_STATION_ALIASES = {
    "Sonderborg": "Sonderborg",
    "Sønderborg": "Sonderborg",
    "Assens": "Assens",
    "Bagenkop": "Bagenkop",
    "Kobenhavn": "Kobenhavn",
    "København": "Kobenhavn",
    "Koge": "Koge",
    "Køge": "Koge",
    "Gedser": "Gedser",
}
_DTU10_CONSTITUENTS = ("M2", "S2", "K1", "O1")
_DTU10_STATION_CACHE = {}


## Dataclasses

Config, split container e result object del pipeline.

In [2]:
@dataclass
class PipelineConfig:
    data_dir: Path = Path("/Users/nicolocaron/Desktop/MASTER PROJECT/data/per_station")
    dtu10_dir: Path = Path("/Volumes/DTU10_TIDEMODEL/OCEAN_TIDE_GRIDS/FES2004_FORMAT")
    use_dtu10_predictor: bool = False
    stations: tuple = tuple(STATIONS.keys())
    base_exog_cols: tuple = ("SLP", "t2m", "u10", "v10")
    optional_prefixes: tuple = ("tide_",)
    horizons: tuple = (1, 3, 6, 12, 24)
    lags: tuple = (0, 1, 2, 3, 6, 12, 24, 48, 72, 96)
    train_frac: float = 0.8
    ridge_alphas: tuple = (0.1, 1.0, 10.0, 100.0)
    target_mode: str = "bias_corrected_error"
    interpolate_exog_limit: int = 6
    model_col: str = "forcoast_p82_m"
    obs_col: str = "tg_obs_m"
    time_col: str = "time"


@dataclass
class EventConfig:
    candidate_threshold_m: float = 0.0
    extreme_quantile: float = 0.95
    gap_h: int = 12
    pad_before_h: int = 12
    pad_after_h: int = 12


@dataclass
class SplitData:
    station: str
    horizon_h: int
    feature_cols: list
    ridge_alphas: tuple
    train_df: pd.DataFrame
    test_df: pd.DataFrame
    X_train: pd.DataFrame
    X_test: pd.DataFrame
    y_train: pd.Series
    y_test: pd.Series
    bias_train: float
    use_dtu10_predictor: bool


@dataclass
class ModelRunResult:
    station: str
    horizon_h: int
    model_name: str
    strict_pure_exog: bool
    use_dtu10_predictor: bool
    estimator: object
    eval_df: pd.DataFrame
    metrics: dict
    coefficients: pd.DataFrame
    diagnostics: dict
    best_alpha: float | None
    bias_train: float
    event_threshold_m: float
    n_train: int
    n_test: int
    n_features: int
    event_metrics: pd.DataFrame | None = None


## Core Utilities

Helper interni per coerenza temporale, frame evento, metric row e assembly dei risultati.

In [3]:
def _normalise_station_name(station_name):
    key = str(station_name)
    if key not in _STATION_ALIASES:
        raise KeyError(f"Unknown station name: {station_name}")
    return _STATION_ALIASES[key]


def _hourly_consistent(index):
    if not isinstance(index, pd.DatetimeIndex) or len(index) < 2:
        return False
    if not (
        (index.minute == 0).all()
        and (index.second == 0).all()
        and (index.microsecond == 0).all()
        and (index.nanosecond == 0).all()
    ):
        return False
    diffs_h = index.to_series().diff().dropna().dt.total_seconds().div(3600.0)
    if diffs_h.empty:
        return False
    return np.allclose(diffs_h.to_numpy(), np.round(diffs_h.to_numpy()))


def _fill_static_columns(df):
    static_cols = []
    for col in df.columns:
        non_null = df[col].dropna()
        if non_null.empty:
            continue
        if non_null.nunique() == 1:
            static_cols.append(col)
    if static_cols:
        df.loc[:, static_cols] = df.loc[:, static_cols].ffill().bfill()
    return df


def _coerce_event_frame(df):
    frame = df.copy()
    if "target_time" in frame.columns:
        time = pd.to_datetime(frame["target_time"])
    elif "base_time" in frame.columns:
        time = pd.to_datetime(frame["base_time"])
    elif "time" in frame.columns:
        time = pd.to_datetime(frame["time"])
    else:
        time = pd.to_datetime(frame.index)

    if "tg_obs_target_m" in frame.columns:
        obs = frame["tg_obs_target_m"]
    elif "tg_obs_m" in frame.columns:
        obs = frame["tg_obs_m"]
    else:
        raise KeyError("Event detection requires tg_obs_m or tg_obs_target_m")

    out = pd.DataFrame({"time": time, "tg_obs_m": obs})
    out = out.dropna(subset=["time"]).sort_values("time").reset_index(drop=True)
    return out


def _empty_event_table():
    return pd.DataFrame(
        columns=[
            "event_id",
            "start_time",
            "end_time",
            "peak_time",
            "peak_tg_obs_m",
        ]
    )


def _empty_event_metrics():
    return pd.DataFrame(
        columns=[
            "station",
            "horizon_h",
            "model_name",
            "event_id",
            "event_rmse_m",
            "peak_obs_error_m",
            "peak_pred_error_m",
            "peak_timing_error_h",
        ]
    )


def _metric_row(result):
    row = {
        "station": result.station,
        "horizon_h": result.horizon_h,
        "model_name": result.model_name,
        "strict_pure_exog": result.strict_pure_exog,
        "use_dtu10_predictor": result.use_dtu10_predictor,
        "r2": result.metrics.get("r2"),
        "rmse_m": result.metrics.get("rmse_m"),
        "mae_m": result.metrics.get("mae_m"),
        "bias_m": result.metrics.get("bias_m"),
        "n_train": result.n_train,
        "n_test": result.n_test,
        "n_features": result.n_features,
        "event_threshold_m": result.event_threshold_m,
    }
    return row


def _coefficient_map_to_frame(coef_map, result):
    rows = []
    for feature, coef in coef_map.items():
        if feature == "const":
            continue
        if "_lag" not in feature:
            continue
        variable, lag_h = feature.rsplit("_lag", 1)
        rows.append(
            {
                "station": result.station,
                "horizon_h": result.horizon_h,
                "model_name": result.model_name,
                "variable": variable,
                "lag_h": int(lag_h),
                "coef": float(coef),
            }
        )
    if not rows:
        return pd.DataFrame(
            columns=[
                "station",
                "horizon_h",
                "model_name",
                "variable",
                "lag_h",
                "coef",
                "abs_coef",
                "is_peak_lag",
            ]
        )
    out = pd.DataFrame(rows)
    out["abs_coef"] = out["coef"].abs()
    peak_abs = out.groupby("variable")["abs_coef"].transform("max")
    out["is_peak_lag"] = np.isclose(out["abs_coef"], peak_abs)
    return out.sort_values(["variable", "lag_h"]).reset_index(drop=True)


def _assemble_eval_df(split, pred_eps):
    eval_df = split.test_df.copy()
    eval_df["eps_pred"] = np.asarray(pred_eps, dtype=float)
    eval_df["corrected_model_pred_m"] = (
        eval_df["model_target_m"] - split.bias_train - eval_df["eps_pred"]
    )
    eval_df["corrected_residual_m"] = (
        eval_df["corrected_model_pred_m"] - eval_df["tg_obs_target_m"]
    )
    return eval_df


def _make_model_result(split, model_name, estimator, pred_eps, coef_map, diagnostics, strict_pure_exog, best_alpha=None):
    eval_df = _assemble_eval_df(split, pred_eps)
    result = ModelRunResult(
        station=split.station,
        horizon_h=split.horizon_h,
        model_name=model_name,
        strict_pure_exog=strict_pure_exog,
        use_dtu10_predictor=split.use_dtu10_predictor,
        estimator=estimator,
        eval_df=eval_df,
        metrics=compute_metrics(eval_df),
        coefficients=pd.DataFrame(),
        diagnostics=diagnostics,
        best_alpha=best_alpha,
        bias_train=float(split.bias_train),
        event_threshold_m=np.nan,
        n_train=int(len(split.train_df)),
        n_test=int(len(split.test_df)),
        n_features=int(len(split.feature_cols)),
        event_metrics=_empty_event_metrics(),
    )
    diagnostics["coef_map"] = pd.Series(coef_map, dtype=float)
    result.coefficients = extract_linear_coefficients(result)
    return result


## DTU10 Helpers

Reader FES2004-format, cache delle costanti armoniche e ricostruzione opzionale del tide predictor.

In [4]:
def _modified_julian_day(index):
    return index.to_julian_date().to_numpy() - 2400000.5


def _extract_fes_chunked_constants(asc_path, lat, lon):
    lon = float(lon)
    if lon < 0:
        lon = lon + 360.0

    with Path(asc_path).open("r") as f:
        lon_min, lon_max = map(float, f.readline().split())
        lat_min, lat_max = map(float, f.readline().split())
        dlon, dlat = map(float, f.readline().split())
        nlon, nlat = map(int, f.readline().split())
        missing_amp, missing_phase = map(float, f.readline().split())

        lon = min(max(lon, lon_min), lon_max)
        lat = min(max(float(lat), lat_min), lat_max)

        lon_pos = (lon - lon_min) / dlon
        lat_pos = (lat - lat_min) / dlat
        i0 = int(np.floor(lon_pos))
        j0 = int(np.floor(lat_pos))
        i1 = min(i0 + 1, nlon - 1)
        j1 = min(j0 + 1, nlat - 1)
        needed_rows = {j0, j1}
        needed_cols = {i0, i1}

        samples = {}
        row = 0
        col = 0
        while row < nlat:
            amp_line = f.readline()
            while amp_line and not amp_line.strip():
                amp_line = f.readline()
            if not amp_line:
                break

            phase_line = f.readline()
            while phase_line and not phase_line.strip():
                phase_line = f.readline()
            if not phase_line:
                raise ValueError(f"Incomplete amplitude/phase pair in {asc_path}")

            amp_chunk = np.fromstring(amp_line, sep=" ")
            phase_chunk = np.fromstring(phase_line, sep=" ")
            if amp_chunk.size != phase_chunk.size:
                raise ValueError(f"Mismatched chunk lengths in {asc_path}")

            next_col = col + int(amp_chunk.size)
            if row in needed_rows:
                for target_col in needed_cols:
                    if col <= target_col < next_col:
                        local_idx = target_col - col
                        samples[(row, target_col)] = (
                            float(amp_chunk[local_idx]),
                            float(phase_chunk[local_idx]),
                        )

            col = next_col
            if col == nlon:
                row += 1
                col = 0
            elif col > nlon:
                raise ValueError(f"Longitude overflow while parsing {asc_path}")

        if row != nlat:
            raise ValueError(f"Unexpected end-of-file while parsing {asc_path}")

    corners = {}
    for row_idx in (j0, j1):
        for col_idx in (i0, i1):
            amp, phase = samples.get((row_idx, col_idx), (np.nan, np.nan))
            if np.isclose(amp, missing_amp):
                amp = np.nan
            if np.isclose(phase, missing_phase):
                phase = np.nan
            corners[(row_idx, col_idx)] = (amp, phase)

    def bilinear(values):
        v00 = values[(j0, i0)]
        v01 = values[(j0, i1)]
        v10 = values[(j1, i0)]
        v11 = values[(j1, i1)]
        tx = lon_pos - i0
        ty = lat_pos - j0
        grid = np.array([v00, v01, v10, v11], dtype=float)
        if np.all(np.isfinite(grid)):
            return (
                (1.0 - tx) * (1.0 - ty) * v00
                + tx * (1.0 - ty) * v01
                + (1.0 - tx) * ty * v10
                + tx * ty * v11
            )

        valid = []
        for row_idx, col_idx in ((j0, i0), (j0, i1), (j1, i0), (j1, i1)):
            value = values[(row_idx, col_idx)]
            if np.isfinite(value):
                dist = np.hypot(lon_pos - col_idx, lat_pos - row_idx)
                valid.append((dist, value))
        if not valid:
            return np.nan
        valid.sort(key=lambda item: item[0])
        return float(valid[0][1])

    amp_values = {key: value[0] for key, value in corners.items()}
    phase_values = {key: value[1] for key, value in corners.items()}
    amp_cm = bilinear(amp_values)
    phase_deg = bilinear(phase_values)
    return amp_cm, phase_deg


def _get_station_dtu10_constants(lat, lon, cfg):
    cache_key = (round(float(lat), 6), round(float(lon), 6), str(Path(cfg.dtu10_dir).expanduser()))
    if cache_key in _DTU10_STATION_CACHE:
        return _DTU10_STATION_CACHE[cache_key]

    constants = {}
    for constituent in _DTU10_CONSTITUENTS:
        asc_path = Path(cfg.dtu10_dir).expanduser() / f"{constituent}DTU10.asc"
        if not asc_path.exists():
            raise FileNotFoundError(f"Missing DTU10 constituent grid: {asc_path}")
        amp_cm, phase_deg = _extract_fes_chunked_constants(asc_path, lat=lat, lon=lon)
        constants[constituent] = {
            "amplitude_m": np.nan if not np.isfinite(amp_cm) else float(amp_cm) / 100.0,
            "phase_deg": float(phase_deg) if np.isfinite(phase_deg) else np.nan,
        }

    _DTU10_STATION_CACHE[cache_key] = constants
    return constants


def _reconstruct_dtu10_tide(index, constants, corrections="FES"):
    import pyTMD.constituents

    constituent_ids = [name.lower() for name in _DTU10_CONSTITUENTS]
    mjd = _modified_julian_day(pd.DatetimeIndex(index))
    pu, pf, G = pyTMD.constituents.arguments(mjd, constituent_ids, corrections=corrections)

    harmonics = np.array(
        [
            constants[name]["amplitude_m"] * np.exp(-1.0j * np.radians(constants[name]["phase_deg"]))
            for name in _DTU10_CONSTITUENTS
        ],
        dtype=np.complex128,
    )

    theta = np.radians(G) + pu
    tide = (
        harmonics.real[np.newaxis, :] * pf * np.cos(theta)
        - harmonics.imag[np.newaxis, :] * pf * np.sin(theta)
    ).sum(axis=1)
    return pd.Series(tide, index=index, name="tide_dtu10_m")


## Data Loading And Dataset Build

Caricamento parquet, rilevamento esogene, preparazione del frame e costruzione del dataset supervised diretto.

In [5]:
def load_station_data(station_name, cfg):
    station_key = _normalise_station_name(station_name)
    station_id = STATIONS[station_key]
    data_dir = Path(cfg.data_dir).expanduser()
    file_path = data_dir / f"station_{station_id}_{station_key}.parquet"
    if not file_path.exists():
        matches = sorted(data_dir.glob(f"station_{station_id}_*.parquet"))
        if not matches:
            raise FileNotFoundError(f"Station parquet not found for {station_name}: {file_path}")
        file_path = matches[0]

    df = pd.read_parquet(file_path)
    df[cfg.time_col] = pd.to_datetime(df[cfg.time_col])
    df = df.sort_values(cfg.time_col).set_index(cfg.time_col)
    df.index.name = cfg.time_col

    if _hourly_consistent(df.index):
        df = df.asfreq("h")
        df = _fill_static_columns(df)

    return df


def detect_exogenous_columns(df, cfg):
    missing = [col for col in cfg.base_exog_cols if col not in df.columns]
    if missing:
        raise KeyError(f"Missing required exogenous columns: {missing}")

    exog_cols = list(cfg.base_exog_cols)
    optional = []
    for prefix in cfg.optional_prefixes:
        optional.extend([col for col in df.columns if str(col).startswith(prefix)])
    optional = sorted(set(optional))

    for col in optional:
        if col == "tide_dtu10_m" and not cfg.use_dtu10_predictor:
            continue
        if df[col].notna().any():
            exog_cols.append(col)

    return exog_cols


def maybe_add_dtu10_tide(df, lat, lon, cfg):
    if not cfg.use_dtu10_predictor:
        return df

    out = df.copy()
    if "tide_dtu10_m" in out.columns and out["tide_dtu10_m"].notna().any():
        return out

    dtu10_dir = Path(cfg.dtu10_dir).expanduser()
    if not dtu10_dir.exists():
        warnings.warn(f"DTU10 directory unavailable, skipping tide_dtu10_m: {dtu10_dir}")
        return out

    try:
        constants = _get_station_dtu10_constants(lat=lat, lon=lon, cfg=cfg)
        out["tide_dtu10_m"] = _reconstruct_dtu10_tide(out.index, constants, corrections="FES")
    except ModuleNotFoundError:
        warnings.warn("pyTMD unavailable, skipping tide_dtu10_m predictor")
    except Exception as exc:
        warnings.warn(f"DTU10 reconstruction failed, skipping tide_dtu10_m predictor: {exc}")
    return out


def prepare_station_frame(df, cfg):
    out = df.copy()
    if "SLP" not in out.columns and "msl" in out.columns:
        warnings.warn("Found msl instead of SLP, renaming msl -> SLP")
        out = out.rename(columns={"msl": "SLP"})

    out["error_raw"] = out[cfg.model_col] - out[cfg.obs_col]

    lat = float(out["lat"].dropna().iloc[0])
    lon = float(out["lon"].dropna().iloc[0])
    out = maybe_add_dtu10_tide(out, lat=lat, lon=lon, cfg=cfg)

    exog_candidates = [col for col in cfg.base_exog_cols if col in out.columns]
    for prefix in cfg.optional_prefixes:
        exog_candidates.extend([col for col in out.columns if str(col).startswith(prefix)])
    exog_candidates = sorted(set(exog_candidates))

    if exog_candidates and cfg.interpolate_exog_limit is not None:
        out.loc[:, exog_candidates] = out.loc[:, exog_candidates].interpolate(
            method="time",
            limit=int(cfg.interpolate_exog_limit),
            limit_direction="both",
        )

    return out


def build_lagged_exog_matrix(df, exog_cols, lags):
    lagged = {}
    for col in exog_cols:
        for lag in sorted(dict.fromkeys(lags)):
            lagged[f"{col}_lag{int(lag)}"] = df[col].shift(int(lag))
    return pd.DataFrame(lagged, index=df.index)


def build_direct_dataset(df, exog_cols, lags, horizon, cfg):
    lagged_x = build_lagged_exog_matrix(df, exog_cols, lags)
    dataset = lagged_x.copy()
    dataset["base_time"] = dataset.index
    dataset["target_time"] = dataset.index + pd.to_timedelta(int(horizon), unit="h")

    target_frame = df[[cfg.obs_col, cfg.model_col, "error_raw"]].rename(
        columns={
            cfg.obs_col: "tg_obs_target_m",
            cfg.model_col: "model_target_m",
            "error_raw": "error_raw_target",
        }
    )
    dataset = dataset.join(target_frame, on="target_time")

    required_cols = [
        "base_time",
        "target_time",
        "tg_obs_target_m",
        "model_target_m",
        "error_raw_target",
    ] + list(lagged_x.columns)
    dataset = dataset.dropna(subset=required_cols).reset_index(drop=True)
    return dataset


def chronological_train_test_split(dataset, train_frac):
    ordered = dataset.sort_values("target_time").reset_index(drop=True)
    split_idx = int(np.floor(len(ordered) * float(train_frac)))
    split_idx = min(max(split_idx, 1), max(len(ordered) - 1, 1))
    train_df = ordered.iloc[:split_idx].copy()
    test_df = ordered.iloc[split_idx:].copy()
    return train_df, test_df


def apply_train_only_bias_correction(train_df, test_df):
    train_out = train_df.copy()
    test_out = test_df.copy()
    bias_train = float(train_out["error_raw_target"].mean())
    train_out["eps_target"] = train_out["error_raw_target"] - bias_train
    test_out["eps_target"] = test_out["error_raw_target"] - bias_train
    train_out["bias_train"] = bias_train
    test_out["bias_train"] = bias_train
    return train_out, test_out, bias_train


## Model Fitting

CV temporale interna, OLS, Ridge e SARIMAX strict pure-exogenous.

In [6]:
def make_inner_cv_splitter(train_len, horizon, cfg):
    initial_window = max(500, int(0.5 * int(train_len)))
    initial_window = min(initial_window, max(int(train_len) - max(int(horizon), 1) - 1, 1))
    splitter = ExpandingWindowSplitter(
        fh=[int(horizon)],
        initial_window=initial_window,
        step_length=int(horizon),
    )
    try:
        n_folds = len(list(splitter.split(np.arange(int(train_len)))))
    except Exception:
        n_folds = 0
    if n_folds >= 2:
        return splitter

    fallback = ExpandingWindowSplitter(
        fh=[int(horizon)],
        initial_window=initial_window,
        step_length=max(1, int(horizon)),
    )
    return fallback


def fit_ols_exog(split):
    X_train = sm.add_constant(split.X_train, has_constant="add")
    X_test = sm.add_constant(split.X_test, has_constant="add")
    fitted = sm.OLS(split.y_train, X_train).fit(cov_type="HC3")
    pred_eps = fitted.predict(X_test)
    coef_map = fitted.params.reindex(["const"] + split.feature_cols).dropna().to_dict()
    diagnostics = {
        "cov_type": "HC3",
        "params": fitted.params.copy(),
        "bse": fitted.bse.copy(),
        "pvalues": fitted.pvalues.copy(),
        "rsquared_train": float(fitted.rsquared),
    }
    return _make_model_result(
        split=split,
        model_name="ols_exog",
        estimator=fitted,
        pred_eps=pred_eps,
        coef_map=coef_map,
        diagnostics=diagnostics,
        strict_pure_exog=True,
    )


def fit_ridge_exog(split):
    X_train = split.X_train.copy()
    y_train = split.y_train.copy()
    splitter = make_inner_cv_splitter(len(X_train), split.horizon_h, None)
    alpha_scores = {}
    splits = list(splitter.split(np.arange(len(X_train))))

    if not splits:
        holdout_start = max(int(len(X_train) * 0.8), 1)
        splits = [(np.arange(holdout_start), np.arange(holdout_start, len(X_train)))]

    for alpha in split.ridge_alphas:
        alpha_scores[float(alpha)] = []

    for alpha in split.ridge_alphas:
        for train_idx, valid_idx in splits:
            if len(valid_idx) == 0 or len(train_idx) == 0:
                continue
            scaler = StandardScaler()
            scaler.fit(X_train.iloc[train_idx])
            X_tr = scaler.transform(X_train.iloc[train_idx])
            X_va = scaler.transform(X_train.iloc[valid_idx])
            model = Ridge(alpha=float(alpha))
            model.fit(X_tr, y_train.iloc[train_idx])
            pred = model.predict(X_va)
            rmse = float(np.sqrt(mean_squared_error(y_train.iloc[valid_idx], pred)))
            alpha_scores[float(alpha)].append(rmse)

    if not alpha_scores:
        alpha_scores = {float(alpha): [] for alpha in split.ridge_alphas}

    mean_scores = {
        alpha: (np.mean(scores) if scores else np.inf)
        for alpha, scores in alpha_scores.items()
    }
    best_alpha = min(mean_scores, key=mean_scores.get)

    scaler = StandardScaler()
    scaler.fit(X_train)
    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(split.X_test)
    model = Ridge(alpha=float(best_alpha))
    model.fit(X_train_scaled, y_train)
    pred_eps = model.predict(X_test_scaled)

    scale = np.where(scaler.scale_ == 0.0, 1.0, scaler.scale_)
    coef_unscaled = model.coef_ / scale
    intercept_unscaled = float(model.intercept_ - np.sum((scaler.mean_ / scale) * model.coef_))
    coef_map = {"const": intercept_unscaled}
    coef_map.update(dict(zip(split.feature_cols, coef_unscaled)))
    diagnostics = {
        "scaler_mean": pd.Series(scaler.mean_, index=split.feature_cols),
        "scaler_scale": pd.Series(scale, index=split.feature_cols),
        "cv_rmse_by_alpha": mean_scores,
        "coef_scaled": pd.Series(model.coef_, index=split.feature_cols),
        "intercept_scaled": float(model.intercept_),
    }
    return _make_model_result(
        split=split,
        model_name="ridge_exog",
        estimator={"model": model, "scaler": scaler},
        pred_eps=pred_eps,
        coef_map=coef_map,
        diagnostics=diagnostics,
        strict_pure_exog=True,
        best_alpha=float(best_alpha),
    )


def fit_sarimax_exog(split):
    model = SARIMAX(
        endog=split.y_train,
        exog=split.X_train,
        order=(0, 0, 0),
        trend="c",
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    fitted = model.fit(disp=False)
    pred_eps = fitted.get_forecast(steps=len(split.test_df), exog=split.X_test).predicted_mean

    coef_map = {}
    for name in ["intercept"] + split.feature_cols:
        if name in fitted.params.index:
            coef_map["const" if name == "intercept" else name] = float(fitted.params[name])
    diagnostics = {
        "aic": float(fitted.aic),
        "bic": float(fitted.bic),
        "params": fitted.params.copy(),
    }
    return _make_model_result(
        split=split,
        model_name="sarimax_exog",
        estimator=fitted,
        pred_eps=pred_eps,
        coef_map=coef_map,
        diagnostics=diagnostics,
        strict_pure_exog=True,
    )


## Events And Metrics

Detection degli eventi, soglia p95 train-only, metriche globali e per-evento, coefficient extraction.

In [7]:
def detect_candidate_events(df, event_cfg):
    frame = _coerce_event_frame(df)
    exceed = frame.loc[frame["tg_obs_m"] > float(event_cfg.candidate_threshold_m)].copy()
    if exceed.empty:
        return _empty_event_table()

    gap = exceed["time"].diff().dt.total_seconds().div(3600.0).fillna(np.inf)
    exceed["event_id"] = (gap > float(event_cfg.gap_h)).cumsum()

    rows = []
    for event_id, group in exceed.groupby("event_id"):
        peak_idx = group["tg_obs_m"].idxmax()
        rows.append(
            {
                "event_id": int(event_id),
                "start_time": group["time"].min(),
                "end_time": group["time"].max(),
                "peak_time": group.loc[peak_idx, "time"],
                "peak_tg_obs_m": float(group.loc[peak_idx, "tg_obs_m"]),
            }
        )
    return pd.DataFrame(rows)


def compute_train_event_peak_threshold(train_df, event_cfg):
    events = detect_candidate_events(train_df, event_cfg)
    if events.empty:
        return np.nan
    return float(events["peak_tg_obs_m"].quantile(float(event_cfg.extreme_quantile)))


def detect_extreme_events(df, threshold_m, event_cfg):
    if not np.isfinite(threshold_m):
        return _empty_event_table()

    frame = _coerce_event_frame(df)
    exceed = frame.loc[frame["tg_obs_m"] > float(threshold_m)].copy()
    if exceed.empty:
        return _empty_event_table()

    gap = exceed["time"].diff().dt.total_seconds().div(3600.0).fillna(np.inf)
    exceed["event_id"] = (gap > float(event_cfg.gap_h)).cumsum()

    rows = []
    for event_id, group in exceed.groupby("event_id"):
        peak_idx = group["tg_obs_m"].idxmax()
        rows.append(
            {
                "event_id": int(event_id),
                "start_time": group["time"].min() - pd.Timedelta(hours=int(event_cfg.pad_before_h)),
                "end_time": group["time"].max() + pd.Timedelta(hours=int(event_cfg.pad_after_h)),
                "peak_time": group.loc[peak_idx, "time"],
                "peak_tg_obs_m": float(group.loc[peak_idx, "tg_obs_m"]),
            }
        )
    return pd.DataFrame(rows)


def filter_event_rows(dataset, event_windows):
    if event_windows is None or event_windows.empty:
        out = dataset.head(0).copy()
        out["event_id"] = pd.Series(dtype=int)
        return out

    pieces = []
    for row in event_windows.itertuples(index=False):
        mask = (
            (pd.to_datetime(dataset["target_time"]) >= row.start_time)
            & (pd.to_datetime(dataset["target_time"]) <= row.end_time)
        )
        subset = dataset.loc[mask].copy()
        if subset.empty:
            continue
        subset["event_id"] = int(row.event_id)
        subset["event_start_time"] = row.start_time
        subset["event_end_time"] = row.end_time
        subset["event_peak_time"] = row.peak_time
        subset["event_peak_tg_obs_m"] = float(row.peak_tg_obs_m)
        pieces.append(subset)

    if not pieces:
        out = dataset.head(0).copy()
        out["event_id"] = pd.Series(dtype=int)
        return out
    return pd.concat(pieces, ignore_index=True)


def compute_metrics(eval_df):
    metrics = {"r2": np.nan, "rmse_m": np.nan, "mae_m": np.nan, "bias_m": np.nan}
    valid = eval_df.dropna(subset=["eps_target", "eps_pred"])
    if valid.empty:
        return metrics

    y_true = valid["eps_target"].to_numpy(dtype=float)
    y_pred = valid["eps_pred"].to_numpy(dtype=float)
    metrics["rmse_m"] = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    metrics["mae_m"] = float(mean_absolute_error(y_true, y_pred))
    metrics["bias_m"] = float(np.mean(y_pred - y_true))
    if len(valid) >= 2:
        metrics["r2"] = float(r2_score(y_true, y_pred))
    return metrics


def compute_event_metrics(eval_df):
    if "event_id" not in eval_df.columns or eval_df.empty:
        return _empty_event_metrics()

    rows = []
    for event_id, group in eval_df.groupby("event_id"):
        group = group.dropna(subset=["eps_target", "eps_pred", "target_time"]).sort_values("target_time")
        if group.empty:
            continue
        y_true = group["eps_target"].to_numpy(dtype=float)
        y_pred = group["eps_pred"].to_numpy(dtype=float)
        obs_peak_idx = int(np.nanargmax(np.abs(y_true)))
        pred_peak_idx = int(np.nanargmax(np.abs(y_pred)))
        peak_timing_error_h = (
            pd.Timestamp(group.iloc[pred_peak_idx]["target_time"])
            - pd.Timestamp(group.iloc[obs_peak_idx]["target_time"])
        ).total_seconds() / 3600.0

        rows.append(
            {
                "station": group["station"].iloc[0] if "station" in group.columns else None,
                "horizon_h": group["horizon_h"].iloc[0] if "horizon_h" in group.columns else None,
                "model_name": group["model_name"].iloc[0] if "model_name" in group.columns else None,
                "event_id": int(event_id),
                "event_rmse_m": float(np.sqrt(mean_squared_error(y_true, y_pred))),
                "peak_obs_error_m": float(y_true[obs_peak_idx]),
                "peak_pred_error_m": float(y_pred[pred_peak_idx]),
                "peak_timing_error_h": float(peak_timing_error_h),
            }
        )

    if not rows:
        return _empty_event_metrics()
    return pd.DataFrame(rows)


def extract_linear_coefficients(result):
    coef_map = result.diagnostics.get("coef_map")
    if coef_map is None:
        return _coefficient_map_to_frame({}, result)
    if isinstance(coef_map, pd.Series):
        coef_map = coef_map.to_dict()
    return _coefficient_map_to_frame(coef_map, result)


## Orchestration

Suite per stazione-orizzonte e batch run su tutte le stazioni.

In [8]:
def run_station_horizon_suite(station_name, horizon, cfg, event_cfg):
    station_key = _normalise_station_name(station_name)
    df = load_station_data(station_key, cfg)
    df = prepare_station_frame(df, cfg)
    exog_cols = detect_exogenous_columns(df, cfg)
    dataset = build_direct_dataset(df, exog_cols, cfg.lags, horizon, cfg)
    if dataset.empty:
        raise ValueError(f"No usable rows for {station_key} horizon={horizon}")

    feature_cols = [col for col in dataset.columns if "_lag" in col]
    train_df, test_df = chronological_train_test_split(dataset, cfg.train_frac)
    train_df, test_df, bias_train = apply_train_only_bias_correction(train_df, test_df)

    used_dtu10 = any(col.startswith("tide_dtu10_m_lag") for col in feature_cols)
    split = SplitData(
        station=station_key,
        horizon_h=int(horizon),
        feature_cols=feature_cols,
        ridge_alphas=tuple(float(alpha) for alpha in cfg.ridge_alphas),
        train_df=train_df,
        test_df=test_df,
        X_train=train_df[feature_cols],
        X_test=test_df[feature_cols],
        y_train=train_df["eps_target"],
        y_test=test_df["eps_target"],
        bias_train=float(bias_train),
        use_dtu10_predictor=bool(used_dtu10),
    )

    event_threshold_m = compute_train_event_peak_threshold(train_df, event_cfg)
    train_event_windows = detect_extreme_events(train_df, event_threshold_m, event_cfg)
    test_event_windows = detect_extreme_events(test_df, event_threshold_m, event_cfg)

    results = {}
    for fit_func in (fit_ols_exog, fit_ridge_exog, fit_sarimax_exog):
        result = fit_func(split)
        result.eval_df["station"] = station_key
        result.eval_df["horizon_h"] = int(horizon)
        result.eval_df["model_name"] = result.model_name
        result.event_threshold_m = float(event_threshold_m) if np.isfinite(event_threshold_m) else np.nan
        event_eval_df = filter_event_rows(result.eval_df, test_event_windows)
        result.diagnostics["event_eval_df"] = event_eval_df
        result.event_metrics = compute_event_metrics(event_eval_df)
        results[result.model_name] = result

    train_event_rows = filter_event_rows(train_df, train_event_windows)
    test_event_rows = filter_event_rows(test_df, test_event_windows)
    if (
        not train_event_rows.empty
        and not test_event_rows.empty
        and len(train_event_rows) >= max(len(feature_cols), 20)
    ):
        train_event_rows = train_event_rows.copy()
        test_event_rows = test_event_rows.copy()
        aux_split = SplitData(
            station=station_key,
            horizon_h=int(horizon),
            feature_cols=feature_cols,
            ridge_alphas=tuple(float(alpha) for alpha in cfg.ridge_alphas),
            train_df=train_event_rows,
            test_df=test_event_rows,
            X_train=train_event_rows[feature_cols],
            X_test=test_event_rows[feature_cols],
            y_train=train_event_rows["eps_target"],
            y_test=test_event_rows["eps_target"],
            bias_train=float(bias_train),
            use_dtu10_predictor=bool(used_dtu10),
        )
        aux_result = fit_ridge_exog(aux_split)
        aux_result.model_name = "ridge_event_only_aux"
        aux_result.strict_pure_exog = False
        aux_result.eval_df["station"] = station_key
        aux_result.eval_df["horizon_h"] = int(horizon)
        aux_result.eval_df["model_name"] = aux_result.model_name
        aux_result.metrics = compute_metrics(aux_result.eval_df)
        aux_result.event_threshold_m = float(event_threshold_m) if np.isfinite(event_threshold_m) else np.nan
        aux_result.diagnostics["event_eval_df"] = aux_result.eval_df.copy()
        aux_result.event_metrics = compute_event_metrics(aux_result.eval_df.assign(
            station=station_key,
            horizon_h=int(horizon),
            model_name=aux_result.model_name,
        ))
        aux_result.coefficients = extract_linear_coefficients(aux_result)
        results[aux_result.model_name] = aux_result

    metrics_df = pd.DataFrame([_metric_row(result) for result in results.values()])
    event_metrics_df = pd.concat(
        [result.event_metrics for result in results.values() if result.event_metrics is not None],
        ignore_index=True,
    ) if results else _empty_event_metrics()
    coefficients_df = pd.concat(
        [result.coefficients for result in results.values() if not result.coefficients.empty],
        ignore_index=True,
    ) if results else pd.DataFrame()

    return {
        "station": station_key,
        "horizon_h": int(horizon),
        "prepared_frame": df,
        "dataset": dataset,
        "train_df": train_df,
        "test_df": test_df,
        "event_threshold_m": float(event_threshold_m) if np.isfinite(event_threshold_m) else np.nan,
        "train_event_windows": train_event_windows,
        "test_event_windows": test_event_windows,
        "results": results,
        "metrics": metrics_df,
        "event_metrics": event_metrics_df,
        "coefficients": coefficients_df,
        "diagnostics": {name: res.diagnostics for name, res in results.items()},
    }


def run_all_stations(cfg, event_cfg):
    suites = {}
    metrics_parts = []
    event_metrics_parts = []
    coefficient_parts = []
    failures = []

    for station_name in cfg.stations:
        for horizon in cfg.horizons:
            try:
                suite = run_station_horizon_suite(station_name, horizon, cfg, event_cfg)
                suites[(station_name, int(horizon))] = suite
                metrics_parts.append(suite["metrics"])
                if not suite["event_metrics"].empty:
                    event_metrics_parts.append(suite["event_metrics"])
                if not suite["coefficients"].empty:
                    coefficient_parts.append(suite["coefficients"])
            except Exception as exc:
                warnings.warn(f"Failed station={station_name}, horizon={horizon}: {exc}")
                failures.append(
                    {
                        "station": station_name,
                        "horizon_h": int(horizon),
                        "error": str(exc),
                    }
                )

    metrics_df = pd.concat(metrics_parts, ignore_index=True) if metrics_parts else pd.DataFrame()
    event_metrics_df = (
        pd.concat(event_metrics_parts, ignore_index=True)
        if event_metrics_parts
        else _empty_event_metrics()
    )
    coefficients_df = (
        pd.concat(coefficient_parts, ignore_index=True)
        if coefficient_parts
        else pd.DataFrame(
            columns=[
                "station",
                "horizon_h",
                "model_name",
                "variable",
                "lag_h",
                "coef",
                "abs_coef",
                "is_peak_lag",
            ]
        )
    )
    failures_df = pd.DataFrame(failures)

    return {
        "suites": suites,
        "metrics": metrics_df,
        "event_metrics": event_metrics_df,
        "coefficients": coefficients_df,
        "failures": failures_df,
    }


## Plotting

Heatmap, plot evento, diagnostica residui e impulse-response plots.

In [10]:
def plot_metric_heatmap(metrics_df, metric="r2", model_name="ridge_exog", ax=None):
    subset = metrics_df.loc[metrics_df["model_name"] == model_name].copy()
    if subset.empty:
        raise ValueError(f"No metrics available for model_name={model_name}")

    subset["station"] = pd.Categorical(
        subset["station"],
        categories=list(STATIONS.keys()),
        ordered=True,
    )
    pivot = subset.pivot_table(index="station", columns="horizon_h", values=metric, aggfunc="mean")
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 4))

    cmap = "RdBu_r" if metric == "r2" else "mako"
    sns.heatmap(pivot, annot=True, fmt=".3f", cmap=cmap, ax=ax)
    ax.set_title(f"{model_name} | {metric}")
    ax.set_xlabel("Forecast Horizon [h]")
    ax.set_ylabel("Station")
    return ax


def plot_event_window(result, event_id=None):
    if isinstance(result, ModelRunResult):
        eval_df = result.diagnostics.get("event_eval_df", result.eval_df).copy()
        model_name = result.model_name
        station = result.station
        horizon_h = result.horizon_h
    else:
        eval_df = result.copy()
        model_name = "model"
        station = "station"
        horizon_h = "h"

    if "event_id" not in eval_df.columns:
        raise KeyError("plot_event_window requires an event-filtered evaluation frame with event_id")

    available_ids = sorted(pd.Series(eval_df["event_id"]).dropna().unique().tolist())
    if not available_ids:
        raise ValueError("No event rows available to plot")
    if event_id is None:
        event_id = available_ids[0]

    event_df = eval_df.loc[eval_df["event_id"] == event_id].sort_values("target_time")
    if event_df.empty:
        raise ValueError(f"event_id={event_id} not found")

    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True, constrained_layout=True)
    axes[0].plot(event_df["target_time"], event_df["tg_obs_target_m"], label="TG obs", color="black", lw=1.8)
    axes[0].plot(event_df["target_time"], event_df["model_target_m"], label="HDM raw", color="tab:red", lw=1.2, alpha=0.8)
    axes[0].plot(
        event_df["target_time"],
        event_df["corrected_model_pred_m"],
        label="HDM corrected",
        color="tab:blue",
        lw=1.4,
    )
    axes[0].set_ylabel("Sea Level [m]")
    axes[0].legend(loc="best")
    axes[0].set_title(f"{station} | h={horizon_h}h | {model_name} | event {event_id}")

    axes[1].plot(event_df["target_time"], event_df["eps_target"], label="Observed eps", color="black", lw=1.8)
    axes[1].plot(event_df["target_time"], event_df["eps_pred"], label="Predicted eps", color="tab:blue", lw=1.4)
    axes[1].axhline(0.0, color="0.5", ls="--", lw=0.8)
    axes[1].set_ylabel("Bias-Corrected Error [m]")
    axes[1].set_xlabel("Target Time")
    axes[1].legend(loc="best")
    return fig, axes


def plot_residual_diagnostics(result, nlags=48):
    eval_df = result.eval_df if isinstance(result, ModelRunResult) else result.copy()
    residuals = eval_df["corrected_residual_m"].dropna()
    if residuals.empty:
        raise ValueError("No residuals available for diagnostics")

    fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
    axes[0, 0].plot(eval_df["target_time"], eval_df["corrected_residual_m"], color="tab:blue", lw=1.0)
    axes[0, 0].axhline(0.0, color="0.5", ls="--", lw=0.8)
    axes[0, 0].set_title("Corrected Residuals")
    axes[0, 0].set_xlabel("Target Time")
    axes[0, 0].set_ylabel("Residual [m]")

    sns.histplot(residuals, bins=30, kde=True, ax=axes[0, 1], color="tab:blue")
    axes[0, 1].set_title("Residual Distribution")
    axes[0, 1].set_xlabel("Residual [m]")

    qqplot(residuals, line="45", ax=axes[1, 0])
    axes[1, 0].set_title("Q-Q Plot")

    plot_acf(residuals, lags=min(int(nlags), max(len(residuals) - 1, 1)), ax=axes[1, 1])
    dw_stat = durbin_watson(residuals)
    lb_df = acorr_ljungbox(residuals, lags=[min(int(nlags), max(len(residuals) - 1, 1))], return_df=True)
    lb_pvalue = float(lb_df["lb_pvalue"].iloc[0])
    axes[1, 1].set_title(f"ACF | DW={dw_stat:.3f} | Ljung-Box p={lb_pvalue:.3g}")
    return fig, axes


def plot_impulse_responses(data, station=None, horizon_h=None, model_name=None):
    if isinstance(data, ModelRunResult):
        coef_df = data.coefficients.copy()
        station = data.station
        horizon_h = data.horizon_h
        model_name = data.model_name
    else:
        coef_df = data.copy()

    if station is not None:
        coef_df = coef_df.loc[coef_df["station"] == station]
    if horizon_h is not None:
        coef_df = coef_df.loc[coef_df["horizon_h"] == int(horizon_h)]
    if model_name is not None:
        coef_df = coef_df.loc[coef_df["model_name"] == model_name]
    if coef_df.empty:
        raise ValueError("No coefficients available for impulse-response plotting")

    variables = coef_df["variable"].drop_duplicates().tolist()
    fig, axes = plt.subplots(len(variables), 1, figsize=(12, 3.2 * len(variables)), sharex=True, constrained_layout=True)
    if len(variables) == 1:
        axes = [axes]

    for ax, variable in zip(axes, variables):
        subset = coef_df.loc[coef_df["variable"] == variable].sort_values("lag_h")
        ax.plot(subset["lag_h"], subset["coef"], marker="o", lw=1.4, color="tab:blue")
        peak_subset = subset.loc[subset["is_peak_lag"]]
        if not peak_subset.empty:
            ax.scatter(peak_subset["lag_h"], peak_subset["coef"], color="tab:red", zorder=3, label="Peak lag")
            ax.legend(loc="best")
        ax.axhline(0.0, color="0.5", ls="--", lw=0.8)
        ax.set_ylabel(variable)
        ax.set_title(f"{variable} empirical impulse response")

    axes[-1].set_xlabel("Lag [h]")
    return fig, axes
